# Which Wisdom Book Is This Passage From? Supervised Text Classification of Biblical and Asian Religious Texts

**Name:** Arturo Ramos
**Dataset:** *A Study of Asian Religious and Biblical Texts* (UCI Machine Learning Repository, 2019, CC BY 4.0) — https://doi.org/10.24432/C55S4W — file `AllBooks_baseline_DTM_Labelled.csv`

**Problem.** This is a **supervised learning, multi-class classification** task. Each row is a passage (a chapter-sized chunk) from one of eight wisdom books — four biblical (Proverbs, Ecclesiastes, Ecclesiasticus, Book of Wisdom) and four Asian (Upanishads, Yoga Sutras, Buddhist Sutras, Tao Te Ching) — represented by the counts of 8,266 words. The target is the book the passage comes from. The goal is to train a model that identifies the source book of an unseen passage from its vocabulary and to evaluate it honestly on data it never saw.

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

DATA_DIR = Path("data")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42

print("numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

## 2. Load and Inspect the Dataset

In [ ]:
raw = pd.read_csv(DATA_DIR / "AllBooks_baseline_DTM_Labelled.csv")
print("shape:", raw.shape)
raw.iloc[:5, :8]

In [ ]:
print("First column name:", repr(raw.columns[0]))
print("Example labels:", raw.iloc[:3, 0].tolist(), "...", raw.iloc[-2:, 0].tolist())
print()
print("Column data types:")
print(raw.dtypes.value_counts())
print()
print("Missing values in the whole table:", int(raw.isna().sum().sum()))

In [ ]:
counts = raw.iloc[:, 1:]
labels = raw.iloc[:, 0].str.split("_").str[0]
tokens = counts.sum(axis=1)

print("Passages per book (raw labels):")
print(labels.value_counts().to_string())
print()
print(f"sparsity (share of zero cells): {(counts == 0).to_numpy().mean():.4f}")
print(f"words that appear in only one passage: {int(((counts > 0).sum() == 1).sum()):,} of {counts.shape[1]:,}")
print(f"passages with zero words: {int((tokens == 0).sum())} -> {raw.loc[tokens == 0, raw.columns[0]].tolist()}")
print(f"words per passage: min {tokens.min()}, median {tokens.median():.0f}, max {tokens.max()}")
print()
print("median words per passage, by book:")
print(tokens.groupby(labels).median().sort_values().to_string())

**Data quality notes**

- The table has 590 rows and 8,266 columns: an unnamed first column with a label such as `Buddhism_Ch1`, and 8,266 integer word counts. There are no missing values and all counts are non-negative integers, so no imputation is needed.
- The target is not a column of its own; it has to be parsed from the label (the part before `_Ch`). One of the book names is misspelled in the source (`BookOfEccleasiasticus`).
- One passage (`Buddhism_Ch14`) contains no words at all: it carries no information and is removed.
- The classes are strongly imbalanced, from 189 passages of the Yoga Sutras down to 12 of Ecclesiastes. Accuracy alone would reward a model that ignores the small biblical books.
- The matrix is extremely sparse (99.1 % zeros) and 3,872 words occur in a single passage, which is a high-dimensional, small-sample setting (8,266 features for 589 usable passages) where regularization matters.

## 3. Data Preparation and Preprocessing

In [ ]:
BOOK_NAMES = {
    "BookOfEccleasiasticus": "Ecclesiasticus",   # misspelled in the source file
    "BookOfEcclesiastes": "Ecclesiastes",
    "BookOfProverb": "Proverbs",
    "BookOfWisdom": "Book of Wisdom",
    "Buddhism": "Buddhist Sutras",
    "TaoTeChing": "Tao Te Ching",
    "Upanishad": "Upanishads",
    "YogaSutra": "Yoga Sutras",
}
BIBLICAL = {"Ecclesiasticus", "Ecclesiastes", "Proverbs", "Book of Wisdom"}


def split_features_and_target(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    """Split the raw table into word-count features, a clean book label and the chunk id.

    The label column looks like ``"BookOfProverb_Ch3"``; the book is the part before
    ``"_Ch"`` and is mapped to a readable name (which also fixes the source's misspelling
    of Ecclesiasticus). Every label must map to a known book.
    """
    chunk_id = df.iloc[:, 0].rename("chunk_id")
    book = chunk_id.str.split("_").str[0].map(BOOK_NAMES).rename("book")
    assert book.notna().all(), "unknown book label in the source file"
    X = df.iloc[:, 1:].astype(int)
    return X, book, chunk_id


def drop_empty_passages(X: pd.DataFrame, *others: pd.Series) -> tuple:
    """Remove passages whose word counts are all zero (they carry no information)."""
    keep = X.sum(axis=1) > 0
    return (X.loc[keep],) + tuple(o.loc[keep] for o in others)

In [ ]:
X_all, y_all, ids_all = split_features_and_target(raw)
X, y, ids = drop_empty_passages(X_all, y_all, ids_all)
print("features:", X.shape, "| target:", y.shape)
print(y.value_counts().to_string())

In [ ]:
# Hold out 25 % of the passages as a test set, stratified so every book keeps its proportion.
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, ids, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
print("train:", X_train.shape, "| test:", X_test.shape)
pd.DataFrame({"train": y_train.value_counts(), "test": y_test.value_counts()})

**What was done and why**

- **Target encoding.** The book is parsed from the label and given a readable name; scikit-learn classifiers accept string labels directly, so no numeric encoding is required.
- **Empty passage removed.** A passage with no words cannot be classified from its words.
- **Stratified train/test split (75/25).** With 12 Ecclesiastes passages, a plain random split could leave a book almost absent from the test set; stratification keeps 3 of them in the test set and 9 in training.
- **Feature scaling: TF-IDF instead of raw counts.** Passages differ a lot in length (Upanishad passages have a median of 26 words, while the four biblical books have medians between 236 and 296), so raw counts would mostly encode length. The TF-IDF transform uses sublinear term frequency (1 + log count), down-weights words that occur in many passages, and normalizes every passage to unit length. It is placed **inside a scikit-learn Pipeline** so that the document frequencies are learned from the training folds only, which prevents information from the validation and test passages leaking into training.
- **No feature selection by hand.** Rare words are kept; the model's regularization decides how much each word counts, and the regularization strength is tuned by cross-validation.

## 4. Model Selection and Training

Three models are compared with repeated stratified cross-validation (5 folds × 5 repetitions = 25 fits per setting) **on the training set only**, using the macro-averaged F1 score (the mean of the per-book F1 scores, so each book counts equally):

1. **Baseline** — always predicts the most frequent book. Any real model must beat it.
2. **Multinomial Naive Bayes** on raw counts — a classic, fast text classifier; the smoothing parameter `alpha` is tuned.
3. **Logistic Regression** on TF-IDF features — a linear model with L2 regularization and `class_weight="balanced"` (errors on small books cost more); the inverse regularization strength `C` is tuned.

In [ ]:
# Repeating the 5-fold split 5 times averages out the luck of a single fold assignment,
# which matters when the smallest book has only 9 training passages.
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)

candidates = {
    "Baseline (most frequent)": (DummyClassifier(strategy="most_frequent"), {}),
    "Multinomial Naive Bayes": (MultinomialNB(), {"alpha": [0.01, 0.1, 0.5, 1.0]}),
    "Logistic Regression (TF-IDF)": (
        Pipeline([
            ("tfidf", TfidfTransformer(sublinear_tf=True)),
            ("clf", LogisticRegression(max_iter=5000, class_weight="balanced")),
        ]),
        {"clf__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10]},
    ),
}

searches, rows = {}, []
for name, (estimator, grid) in candidates.items():
    search = GridSearchCV(estimator, grid, cv=cv, scoring="f1_macro", return_train_score=True)
    search.fit(X_train, y_train)
    best = search.best_index_
    searches[name] = search
    rows.append({
        "model": name,
        "best params": search.best_params_ or "-",
        "CV macro-F1 (mean)": search.cv_results_["mean_test_score"][best],
        "CV macro-F1 (sd)": search.cv_results_["std_test_score"][best],
        "train macro-F1": search.cv_results_["mean_train_score"][best],
    })
cv_table = pd.DataFrame(rows).set_index("model").round(3)
cv_table

In [ ]:
# How the regularization strength C affects cross-validated performance
lr_search = searches["Logistic Regression (TF-IDF)"]
pd.DataFrame({
    "C": lr_search.cv_results_["param_clf__C"],
    "CV macro-F1 (mean)": lr_search.cv_results_["mean_test_score"],
    "CV macro-F1 (sd)": lr_search.cv_results_["std_test_score"],
    "train macro-F1": lr_search.cv_results_["mean_train_score"],
}).round(3)

In [ ]:
# Figure 1 — cross-validated comparison of the three models
fig, ax = plt.subplots(figsize=(9, 4.8))
order = cv_table.index.tolist()
ax.barh(order, cv_table["CV macro-F1 (mean)"], xerr=cv_table["CV macro-F1 (sd)"],
        color=["#BBBBBB", "#8172B3", "#4C72B0"], capsize=5)
for i, (m, s) in enumerate(zip(cv_table["CV macro-F1 (mean)"], cv_table["CV macro-F1 (sd)"])):
    ax.text(m + s + 0.015, i, f"{m:.3f}", va="center")
ax.set_xlim(0, 1.1)
ax.invert_yaxis()
ax.set_title("Figure 1. Cross-validated macro-F1 on the training set (5 folds x 5 repeats; error bars = 1 SD)")
ax.set_xlabel("Macro-averaged F1 score")
ax.set_ylabel("Model")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

**Training results.** Logistic regression on TF-IDF features is the best model in repeated cross-validation (mean macro-F1 0.920, SD 0.041), ahead of Naive Bayes (0.872) and far above the baseline (0.061), which only ever predicts the Yoga Sutras. Both real models reach a training score close to 1 (0.999 and 1.000), so they fit the training passages almost perfectly; with 8,266 features and 441 training passages that is expected, and the gap to the cross-validated score is the variance that regularization has to control. The search chose the strongest regularization tested (C = 0.01), but C = 0.1 scores 0.919 and C = 0.03 scores 0.916, so the model is not sensitive to the exact value within that range; weaker regularization (C of 1 or more) lowers the score to about 0.89. The standard deviations of 0.04–0.06 across folds show how much a single passage of a small book can move the macro-F1. The tuned logistic regression is selected and `GridSearchCV` refits it on the whole training set.

## 5. Evaluation on the Held-Out Test Set

In [ ]:
model = searches["Logistic Regression (TF-IDF)"].best_estimator_
y_pred = pd.Series(model.predict(X_test), index=y_test.index)

summary = pd.DataFrame({
    name: {
        "accuracy": accuracy_score(y_test, s.best_estimator_.predict(X_test)),
        "balanced accuracy": balanced_accuracy_score(y_test, s.best_estimator_.predict(X_test)),
        "macro-F1": f1_score(y_test, s.best_estimator_.predict(X_test), average="macro"),
    }
    for name, s in searches.items()
}).T.round(3)
summary

In [ ]:
print(classification_report(y_test, y_pred, digits=3, zero_division=0))

In [ ]:
# Figure 2 — confusion matrix of the selected model on the test set
book_order = ["Proverbs", "Ecclesiastes", "Ecclesiasticus", "Book of Wisdom",
              "Buddhist Sutras", "Tao Te Ching", "Upanishads", "Yoga Sutras"]
cm = confusion_matrix(y_test, y_pred, labels=book_order)
fig, ax = plt.subplots(figsize=(8.5, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=book_order, yticklabels=book_order,
            linewidths=0.5, linecolor="white", ax=ax)
ax.axhline(4, color="black", lw=1.5)
ax.axvline(4, color="black", lw=1.5)
ax.set_title("Figure 2. Confusion matrix on the test set (148 passages)")
ax.set_xlabel("Predicted book")
ax.set_ylabel("True book")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Every misclassified passage, with its length
errors = pd.DataFrame({
    "passage": ids_test, "true book": y_test, "predicted": y_pred, "words in passage": X_test.sum(axis=1),
}).loc[y_test != y_pred].sort_values("passage")
print(f"misclassified: {len(errors)} of {len(y_test)} test passages")
errors

In [ ]:
# Collapsing the eight books into two groups: does the model ever confuse biblical and Asian texts?
true_group = y_test.isin(BIBLICAL).map({True: "Biblical", False: "Asian"})
pred_group = y_pred.isin(BIBLICAL).map({True: "Biblical", False: "Asian"})
print(f"accuracy at the level of biblical vs Asian: {accuracy_score(true_group, pred_group):.3f}")
print(f"median words per test passage: correct {X_test.sum(axis=1)[y_test == y_pred].median():.0f}, "
      f"misclassified {errors['words in passage'].median():.0f}")

In [ ]:
# Figure 3 — the words with the largest positive coefficients for each book
clf = model.named_steps["clf"]
vocab = np.array(X.columns)
top_words = pd.DataFrame({book: vocab[np.argsort(clf.coef_[k])[::-1][:8]] for k, book in enumerate(clf.classes_)})[book_order]

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax, book in zip(axes.ravel(), book_order):
    k = list(clf.classes_).index(book)
    idx = np.argsort(clf.coef_[k])[::-1][:8]
    ax.barh(vocab[idx][::-1], clf.coef_[k][idx][::-1], color="#4C72B0" if book in BIBLICAL else "#DD8452")
    ax.set_title(book, fontsize=11)
    ax.set_xlabel("Coefficient")
    ax.set_ylabel("Word")
fig.suptitle("Figure 3. Eight most indicative words per book (logistic regression coefficients; blue = biblical, orange = Asian)")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_top_words.png", dpi=150, bbox_inches="tight")
plt.show()
top_words

### How stable is a single test score?

The test set has only 3 Ecclesiastes and 5 Book of Wisdom passages, so one passage more or less correct moves the macro-F1 by several points. To see how much of the test result is luck of the split, the two tuned models are re-trained and re-tested on 20 different stratified 75/25 splits (same hyperparameters, different random seeds).

In [ ]:
lr_params = searches["Logistic Regression (TF-IDF)"].best_params_
nb_params = searches["Multinomial Naive Bayes"].best_params_
stability = []
for seed in range(20):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.25, stratify=y, random_state=seed)
    for name, est in [
        ("Logistic Regression (TF-IDF)", candidates["Logistic Regression (TF-IDF)"][0].set_params(**lr_params)),
        ("Multinomial Naive Bayes", MultinomialNB(**nb_params)),
    ]:
        stability.append({"seed": seed, "model": name, "test macro-F1": f1_score(yb, est.fit(Xa, ya).predict(Xb), average="macro")})
stability = pd.DataFrame(stability)
stability_summary = stability.groupby("model")["test macro-F1"].agg(["mean", "std", "min", "max"]).round(3)
wins = (stability.pivot(index="seed", columns="model", values="test macro-F1")
        .pipe(lambda t: (t["Logistic Regression (TF-IDF)"] > t["Multinomial Naive Bayes"]).sum()))
print(f"logistic regression beats Naive Bayes on {wins} of 20 splits")
stability_summary

In [ ]:
# Figure 4 — distribution of the test macro-F1 over 20 random splits
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=stability, x="test macro-F1", y="model", hue="model", palette=["#4C72B0", "#8172B3"], legend=False, width=0.5, ax=ax)
sns.stripplot(data=stability, x="test macro-F1", y="model", color="black", size=4, alpha=0.6, ax=ax)
ax.set_title("Figure 4. Test macro-F1 over 20 different stratified train/test splits")
ax.set_xlabel("Test macro-F1")
ax.set_ylabel("Model")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig4_split_stability.png", dpi=150, bbox_inches="tight")
plt.show()